### 📈 The problem with retail trading

##### ▶️ Related Quant Guild Videos:

- [The 5 Papers That Built Modern Quant Finance](https://youtu.be/ZwS1gMGegrM)

- [I Bet You've Never Found Alpha (and I Can Prove It)](https://youtu.be/UzTJHs3-eT0)

- [Quant Ranks Retail Trading Mistakes that Blow Up Your Account](https://youtu.be/1mpNxBaBeOw)

- [Non-Stationarity and Why Market Timing Fails](https://youtu.be/7nvjrgqKjJE)

- [Quant Busts 3 Trading Myths with Math](https://youtu.be/wJfIk3VnubE)

- [How to Read Options Chains](https://youtu.be/RrRbz6oXwxE)

###### ______________________________________________________________________________________________________________________________________

##### [⚔️ Play the Markets like an RPG on discourses.io](https://discourses.io/)

##### [🚀 Master your Quantitative Skills with Quant Guild](https://quantguild.com)

##### [📚 Visit the Quant Guild Library for more Jupyter Notebooks](https://github.com/romanmichaelpaolucci/Quant-Guild-Library)

##### [📈 Interactive Brokers for Algorithmic Trading](https://www.interactivebrokers.com/mkt/?src=quantguildY&url=%2Fen%2Fwhyib%2Foverview.php)

##### [👾 Join the Quant Guild Discord Server](discord.com/invite/MJ4FU2c6c3)

---

#### There is edge in this trade

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# --- Parameters ---
np.random.seed(np.random.randint(0, 10000))

symbol = "AAPL"
S0 = 190.00

n_seconds = 3600
candle_seconds = 30
animation_ms = 1000
hourly_vol = 0.006

earnings_second = n_seconds // 2
earnings_jump = 0.01  # Always gap up 1%

# --- Simulate 1-second prices for one hour ---
dt = 1 / n_seconds

rets = np.random.normal(
    loc=0,
    scale=hourly_vol * np.sqrt(dt),
    size=n_seconds
)

prices = S0 * np.exp(np.cumsum(rets))
prices = np.insert(prices, 0, S0)

# --- Apply earnings gap up after the midpoint ---
prices[earnings_second:] *= (1 + earnings_jump)

times = pd.date_range(
    start="2025-01-02 10:00:00",
    periods=n_seconds + 1,
    freq="s"
)

tick_df = pd.DataFrame({"price": prices}, index=times)

# --- Convert ticks into candles ---
candles = tick_df["price"].resample(f"{candle_seconds}s").ohlc().dropna()

earnings_time = times[earnings_second]
gap_direction = "Gap Up"
gap_pct = earnings_jump * 100

# --- Figure ---
fig = go.Figure()

fig.add_trace(
    go.Candlestick(
        x=[candles.index[0]],
        open=[candles["open"].iloc[0]],
        high=[candles["high"].iloc[0]],
        low=[candles["low"].iloc[0]],
        close=[candles["close"].iloc[0]],
        increasing_line_color="#00ff88",
        decreasing_line_color="#ff4b4b",
        increasing_fillcolor="#00ff88",
        decreasing_fillcolor="#ff4b4b"
    )
)

# --- Frames and slider ---
frames = []
slider_steps = []

for i in range(1, len(candles) + 1):
    frame_name = f"frame_{i}"

    frames.append(
        go.Frame(
            data=[
                go.Candlestick(
                    x=candles.index[:i],
                    open=candles["open"].iloc[:i],
                    high=candles["high"].iloc[:i],
                    low=candles["low"].iloc[:i],
                    close=candles["close"].iloc[:i],
                    increasing_line_color="#00ff88",
                    decreasing_line_color="#ff4b4b",
                    increasing_fillcolor="#00ff88",
                    decreasing_fillcolor="#ff4b4b"
                )
            ],
            name=frame_name
        )
    )

    slider_steps.append(
        {
            "args": [
                [frame_name],
                {
                    "frame": {"duration": 0, "redraw": True},
                    "mode": "immediate",
                    "fromcurrent": True
                }
            ],
            "label": str(i),
            "method": "animate"
        }
    )

fig.frames = frames

frame_duration = animation_ms / len(frames)

# --- Styling ---
off_white = "#e0e0e0"

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.1)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white)
)

y_min = candles["low"].min() * 0.995
y_max = candles["high"].max() * 1.005

fig.update_layout(
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=760,
    width=1100,
    margin=dict(t=100, b=210),
    showlegend=False,
    title=dict(
        text=f"{symbol} 1H Earnings Simulation • {gap_direction} {gap_pct:.0f}%",
        x=0.5,
        font=dict(size=24)
    ),
    xaxis_rangeslider_visible=False,
    shapes=[
        dict(
            type="line",
            x0=earnings_time,
            x1=earnings_time,
            y0=y_min,
            y1=y_max,
            xref="x",
            yref="y",
            line=dict(
                color="#ffffff",
                width=2,
                dash="dash"
            )
        )
    ],
    annotations=[
        dict(
            x=earnings_time,
            y=y_max,
            xref="x",
            yref="y",
            text="Earnings",
            showarrow=False,
            yanchor="bottom",
            font=dict(color="#ffffff", size=14)
        )
    ],
    updatemenus=[
        {
            "type": "buttons",
            "buttons": [
                {
                    "label": "▶ Play",
                    "method": "animate",
                    "args": [
                        None,
                        {
                            "frame": {
                                "duration": frame_duration,
                                "redraw": True
                            },
                            "transition": {"duration": 0},
                            "fromcurrent": True,
                            "mode": "immediate"
                        }
                    ]
                },
                {
                    "label": "⏸ Pause",
                    "method": "animate",
                    "args": [
                        [None],
                        {
                            "frame": {
                                "duration": 0,
                                "redraw": True
                            },
                            "mode": "immediate",
                            "fromcurrent": True
                        }
                    ]
                }
            ],
            "direction": "left",
            "showactive": False,
            "x": 0.10,
            "xanchor": "left",
            "y": -0.22,
            "yanchor": "top"
        }
    ],
    sliders=[
        {
            "active": 0,
            "yanchor": "top",
            "xanchor": "left",
            "currentvalue": {
                "font": {"size": 13, "color": off_white},
                "prefix": "Candle: ",
                "visible": True,
                "xanchor": "right"
            },
            "transition": {"duration": 0},
            "pad": {"b": 10, "t": 45},
            "len": 0.78,
            "x": 0.18,
            "y": -0.22,
            "steps": slider_steps
        }
    ]
)

fig.update_xaxes(
    axis_style,
    range=[candles.index[0], candles.index[-1]],
    title_text=f"{candle_seconds}-Second Candles"
)

fig.update_yaxes(
    axis_style,
    range=[y_min, y_max],
    title_text="Price"
)

fig.show()

###### ______________________________________________________________________________________________________________________________________

##### Edge in the time sense is not edge in the ensemble sense

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# --- Parameters ---
np.random.seed(np.random.randint(0, 10000))

symbol = "AAPL"
S0 = 190.00

n_paths = 10
n_seconds = 3600
animation_ms = 1000
hourly_vol = 0.006

earnings_second = n_seconds // 2
gap_up_path = np.random.randint(0, n_paths)

earnings_jumps = np.full(n_paths, -0.01)
earnings_jumps[gap_up_path] = 0.01

times = pd.date_range(
    start="2025-01-02 10:00:00",
    periods=n_seconds + 1,
    freq="s"
)

earnings_time = times[earnings_second]

# --- Simulate generic paths ---
all_paths = []
dt = 1 / n_seconds

for path_id in range(n_paths):
    rets = np.random.normal(
        loc=0,
        scale=hourly_vol * np.sqrt(dt),
        size=n_seconds
    )

    prices = S0 * np.exp(np.cumsum(rets))
    prices = np.insert(prices, 0, S0)

    prices[earnings_second:] *= (1 + earnings_jumps[path_id])

    all_paths.append(prices)

all_paths = np.array(all_paths)

# --- Animation frames ---
frame_skip = 20
frame_indices = list(range(1, n_seconds + 1, frame_skip))

if frame_indices[-1] != n_seconds:
    frame_indices.append(n_seconds)

# --- Figure ---
fig = go.Figure()

for path_id in range(n_paths):
    is_gap_up = path_id == gap_up_path

    fig.add_trace(
        go.Scatter(
            x=[times[0]],
            y=[all_paths[path_id, 0]],
            mode="lines",
            line=dict(
                color="#00ff88" if is_gap_up else "#ff4b4b",
                width=5 if is_gap_up else 2
            ),
            opacity=1.0 if is_gap_up else 0.25,
            showlegend=False
        )
    )

# --- Frames and slider ---
frames = []
slider_steps = []

for frame_num, i in enumerate(frame_indices):
    frame_name = f"frame_{frame_num}"

    frames.append(
        go.Frame(
            data=[
                go.Scatter(
                    x=times[:i + 1],
                    y=all_paths[path_id, :i + 1],
                    mode="lines",
                    line=dict(
                        color="#00ff88" if path_id == gap_up_path else "#ff4b4b",
                        width=5 if path_id == gap_up_path else 2
                    ),
                    opacity=1.0 if path_id == gap_up_path else 0.25
                )
                for path_id in range(n_paths)
            ],
            traces=list(range(n_paths)),
            name=frame_name
        )
    )

    slider_steps.append(
        {
            "args": [
                [frame_name],
                {
                    "frame": {"duration": 0, "redraw": True},
                    "mode": "immediate",
                    "fromcurrent": True
                }
            ],
            "label": str(i),
            "method": "animate"
        }
    )

fig.frames = frames

frame_duration = animation_ms / len(frames)

# --- Styling ---
off_white = "#e0e0e0"

y_min = all_paths.min() * 0.995
y_max = all_paths.max() * 1.005

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.1)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white)
)

fig.update_layout(
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=720,
    width=1100,
    margin=dict(t=100, b=170),
    showlegend=False,
    title=dict(
        text=f"{symbol} Simulated Earnings Paths • 1 Gap Up +1% / {n_paths - 1} Gap Down -1%",
        x=0.5,
        font=dict(size=23)
    ),
    shapes=[
        dict(
            type="line",
            x0=earnings_time,
            x1=earnings_time,
            y0=y_min,
            y1=y_max,
            xref="x",
            yref="y",
            line=dict(color="#ffffff", width=2, dash="dash")
        )
    ],
    annotations=[
        dict(
            x=earnings_time,
            y=y_max,
            xref="x",
            yref="y",
            text="Earnings",
            showarrow=False,
            yanchor="bottom",
            font=dict(color="#ffffff", size=14)
        )
    ],
    updatemenus=[
        {
            "type": "buttons",
            "buttons": [
                {
                    "label": "▶ Play",
                    "method": "animate",
                    "args": [
                        None,
                        {
                            "frame": {"duration": frame_duration, "redraw": True},
                            "transition": {"duration": 0},
                            "fromcurrent": True,
                            "mode": "immediate"
                        }
                    ]
                },
                {
                    "label": "⏸ Pause",
                    "method": "animate",
                    "args": [
                        [None],
                        {
                            "frame": {"duration": 0, "redraw": True},
                            "mode": "immediate",
                            "fromcurrent": True
                        }
                    ]
                }
            ],
            "direction": "left",
            "showactive": False,
            "x": 0.12,
            "xanchor": "left",
            "y": -0.18,
            "yanchor": "top"
        }
    ],
    sliders=[
        {
            "active": 0,
            "yanchor": "top",
            "xanchor": "left",
            "currentvalue": {
                "font": {"size": 13, "color": off_white},
                "prefix": "Second: ",
                "visible": True,
                "xanchor": "right"
            },
            "transition": {"duration": 0},
            "pad": {"b": 10, "t": 45},
            "len": 0.78,
            "x": 0.18,
            "y": -0.18,
            "steps": slider_steps
        }
    ]
)

fig.update_xaxes(
    axis_style,
    range=[times[0], times[-1]],
    title_text="One-Hour Earnings Window"
)

fig.update_yaxes(
    axis_style,
    range=[y_min, y_max],
    title_text="Price"
)

fig.show()

###### ______________________________________________________________________________________________________________________________________

##### The statistical mechanism is amorphous

**What actually generates money?**

Trading, like poker, is not one-to-one and it's a game of incomplete information

The only way to accumulate wealth in this system is with an edge

$$\mathbb{E}[\text{Trade P/L}] > 0$$

In [ ]:
import numpy as np 
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- Simulation setup ---
n_steps = 100
initial_wealth = 1000
bet_size = 25
n_paths = 10

# Positive EV (60% win rate)
pos_ev_paths = np.zeros((n_paths, n_steps))
pos_ev_paths[:, 0] = initial_wealth
for path in range(n_paths):
    pos_ev_wins = np.random.random(n_steps) < 0.60
    for i in range(1, n_steps):
        pos_ev_paths[path, i] = pos_ev_paths[path, i-1] + (bet_size if pos_ev_wins[i] else -bet_size)

# Negative EV (40% win rate)
neg_ev_paths = np.zeros((n_paths, n_steps))
neg_ev_paths[:, 0] = initial_wealth
for path in range(n_paths):
    neg_ev_wins = np.random.random(n_steps) < 0.40
    for i in range(1, n_steps):
        neg_ev_paths[path, i] = neg_ev_paths[path, i-1] + (bet_size if neg_ev_wins[i] else -bet_size)

# Expected Value lines
pos_ev_line = initial_wealth + np.arange(n_steps) * bet_size * (0.60 - 0.40)
neg_ev_line = initial_wealth + np.arange(n_steps) * bet_size * (0.40 - 0.60)

# --- Frames ---
frames = []
for i in range(1, n_steps):
    frame_data = []

    # Left (Positive EV)
    for path in range(n_paths):
        opacity = 1.0 - (path * 0.07)
        frame_data.append(
            go.Scatter(
                x=np.arange(i+1),
                y=pos_ev_paths[path, :i+1],
                mode='lines',
                line=dict(color='green', width=2),
                opacity=opacity,
                xaxis='x', yaxis='y',
                showlegend=False
            )
        )
    # Add expected value line
    frame_data.append(
        go.Scatter(
            x=np.arange(i+1),
            y=pos_ev_line[:i+1],
            mode='lines',
            line=dict(color='lime', dash='dash', width=3),
            xaxis='x', yaxis='y',
            showlegend=False
        )
    )

    # Right (Negative EV)
    for path in range(n_paths):
        opacity = 1.0 - (path * 0.07)
        frame_data.append(
            go.Scatter(
                x=np.arange(i+1),
                y=neg_ev_paths[path, :i+1],
                mode='lines',
                line=dict(color='red', width=2),
                opacity=opacity,
                xaxis='x2', yaxis='y2',
                showlegend=False
            )
        )
    # Add expected value line
    frame_data.append(
        go.Scatter(
            x=np.arange(i+1),
            y=neg_ev_line[:i+1],
            mode='lines',
            line=dict(color='orange', dash='dash', width=3),
            xaxis='x2', yaxis='y2',
            showlegend=False
        )
    )

    frames.append(go.Frame(
        name=f"frame{i}",
        data=frame_data,
        layout=go.Layout(
            xaxis=dict(range=[0, n_steps]),
            xaxis2=dict(range=[0, n_steps]),
        )
    ))

# --- Figure ---
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Positive EV (edge > 0)', 'Negative EV (edge < 0)'),
    column_widths=[0.5, 0.5]
)

# Initial left-side traces
for path in range(n_paths):
    fig.add_trace(
        go.Scatter(
            x=[0], y=[initial_wealth],
            mode='lines',
            line=dict(color='green', width=2),
            opacity=1.0 - path * 0.07,
            name='Positive EV Path' if path == 0 else None,
            showlegend=(path == 0)
        ),
        row=1, col=1
    )

# Initial right-side traces
for path in range(n_paths):
    fig.add_trace(
        go.Scatter(
            x=[0], y=[initial_wealth],
            mode='lines',
            line=dict(color='red', width=2),
            opacity=1.0 - path * 0.07,
            name='Negative EV Path' if path == 0 else None,
            showlegend=(path == 0)
        ),
        row=1, col=2
    )

# Add static expected value lines
fig.add_trace(
    go.Scatter(x=np.arange(n_steps), y=pos_ev_line, mode='lines',
               line=dict(color='lime', dash='dash', width=3),
               name='Expected Value (Positive EV)'),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=np.arange(n_steps), y=neg_ev_line, mode='lines',
               line=dict(color='orange', dash='dash', width=3),
               name='Expected Value (Negative EV)'),
    row=1, col=2
)

# --- Apply frames ---
fig.frames = frames

# --- Layout ---
fig.update_layout(
    height=550,
    width=1100,
    title_text="Wealth Paths: Positive vs Negative Expected Value<br><sup>Dashed line = Theoretical EV</sup>",
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)',
    font=dict(color='white'),
    showlegend=True,
    legend=dict(orientation='h', y=-0.2, x=0.5, xanchor='center'),
    updatemenus=[{
        'type': 'buttons',
        'x': 0.05, 'y': -0.1,
        'showactive': False,
        'buttons': [{
            'label': 'Play',
            'method': 'animate',
            'args': [None, {
                'frame': {'duration': 30, 'redraw': True},
                'fromcurrent': True,
                'transition': {'duration': 0}
            }]
        }]
    }]
)

# --- Axes ---
fig.update_xaxes(title_text="Round", range=[0, n_steps],
                 showgrid=True, gridcolor='rgba(128,128,128,0.3)')
fig.update_yaxes(title_text="Wealth", range=[0, 2000],
                 showgrid=True, gridcolor='rgba(128,128,128,0.3)')

fig.show()


![Nonergodic Wealth Illustration](nonergodic.png)

###### ______________________________________________________________________________________________________________________________________

##### Edge is Amorphous

Quants try to pin it down, *what's your edge*, let's quantify it

 $$
 \mathbb{E}[\text{Trade P/L}] = P(\text{win}) \cdot \mathbb{E}[\mathrm{Trade P\!/\!L}\mid\text{win}] + P(\text{loss}) \cdot \mathbb{E}[\text{Trade P\!/\!L}\mid\text{loss}]
 $$

 Ok so what goes into each of these components? 
 
 **LITERALLY EVERYTHING** because these mathematical objects are a compression of an unfathomable amount of information

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# --- Parameters ---
np.random.seed(np.random.randint(0, 10000))

symbol = "AAPL"
S0 = 190.00

n_seconds = 3600
candle_seconds = 30
animation_ms = 1000
hourly_vol = 0.006

earnings_second = n_seconds // 2
earnings_jump = 0.01  # Always gap up 1%

# --- Simulate 1-second prices for one hour ---
dt = 1 / n_seconds

rets = np.random.normal(
    loc=0,
    scale=hourly_vol * np.sqrt(dt),
    size=n_seconds
)

prices = S0 * np.exp(np.cumsum(rets))
prices = np.insert(prices, 0, S0)

# --- Apply earnings gap up after the midpoint ---
prices[earnings_second:] *= (1 + earnings_jump)

times = pd.date_range(
    start="2025-01-02 10:00:00",
    periods=n_seconds + 1,
    freq="s"
)

tick_df = pd.DataFrame({"price": prices}, index=times)

# --- Convert ticks into candles ---
candles = tick_df["price"].resample(f"{candle_seconds}s").ohlc().dropna()

earnings_time = times[earnings_second]
gap_direction = "Gap Up"
gap_pct = earnings_jump * 100

# --- Figure ---
fig = go.Figure()

fig.add_trace(
    go.Candlestick(
        x=[candles.index[0]],
        open=[candles["open"].iloc[0]],
        high=[candles["high"].iloc[0]],
        low=[candles["low"].iloc[0]],
        close=[candles["close"].iloc[0]],
        increasing_line_color="#00ff88",
        decreasing_line_color="#ff4b4b",
        increasing_fillcolor="#00ff88",
        decreasing_fillcolor="#ff4b4b"
    )
)

# --- Frames and slider ---
frames = []
slider_steps = []

for i in range(1, len(candles) + 1):
    frame_name = f"frame_{i}"

    frames.append(
        go.Frame(
            data=[
                go.Candlestick(
                    x=candles.index[:i],
                    open=candles["open"].iloc[:i],
                    high=candles["high"].iloc[:i],
                    low=candles["low"].iloc[:i],
                    close=candles["close"].iloc[:i],
                    increasing_line_color="#00ff88",
                    decreasing_line_color="#ff4b4b",
                    increasing_fillcolor="#00ff88",
                    decreasing_fillcolor="#ff4b4b"
                )
            ],
            name=frame_name
        )
    )

    slider_steps.append(
        {
            "args": [
                [frame_name],
                {
                    "frame": {"duration": 0, "redraw": True},
                    "mode": "immediate",
                    "fromcurrent": True
                }
            ],
            "label": str(i),
            "method": "animate"
        }
    )

fig.frames = frames

frame_duration = animation_ms / len(frames)

# --- Styling ---
off_white = "#e0e0e0"

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.1)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white)
)

y_min = candles["low"].min() * 0.995
y_max = candles["high"].max() * 1.005

fig.update_layout(
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=760,
    width=1100,
    margin=dict(t=100, b=210),
    showlegend=False,
    title=dict(
        text=f"{symbol} 1H Earnings Simulation • {gap_direction} {gap_pct:.0f}%",
        x=0.5,
        font=dict(size=24)
    ),
    xaxis_rangeslider_visible=False,
    shapes=[
        dict(
            type="line",
            x0=earnings_time,
            x1=earnings_time,
            y0=y_min,
            y1=y_max,
            xref="x",
            yref="y",
            line=dict(
                color="#ffffff",
                width=2,
                dash="dash"
            )
        )
    ],
    annotations=[
        dict(
            x=earnings_time,
            y=y_max,
            xref="x",
            yref="y",
            text="Earnings",
            showarrow=False,
            yanchor="bottom",
            font=dict(color="#ffffff", size=14)
        )
    ],
    updatemenus=[
        {
            "type": "buttons",
            "buttons": [
                {
                    "label": "▶ Play",
                    "method": "animate",
                    "args": [
                        None,
                        {
                            "frame": {
                                "duration": frame_duration,
                                "redraw": True
                            },
                            "transition": {"duration": 0},
                            "fromcurrent": True,
                            "mode": "immediate"
                        }
                    ]
                },
                {
                    "label": "⏸ Pause",
                    "method": "animate",
                    "args": [
                        [None],
                        {
                            "frame": {
                                "duration": 0,
                                "redraw": True
                            },
                            "mode": "immediate",
                            "fromcurrent": True
                        }
                    ]
                }
            ],
            "direction": "left",
            "showactive": False,
            "x": 0.10,
            "xanchor": "left",
            "y": -0.22,
            "yanchor": "top"
        }
    ],
    sliders=[
        {
            "active": 0,
            "yanchor": "top",
            "xanchor": "left",
            "currentvalue": {
                "font": {"size": 13, "color": off_white},
                "prefix": "Candle: ",
                "visible": True,
                "xanchor": "right"
            },
            "transition": {"duration": 0},
            "pad": {"b": 10, "t": 45},
            "len": 0.78,
            "x": 0.18,
            "y": -0.22,
            "steps": slider_steps
        }
    ]
)

fig.update_xaxes(
    axis_style,
    range=[candles.index[0], candles.index[-1]],
    title_text=f"{candle_seconds}-Second Candles"
)

fig.update_yaxes(
    axis_style,
    range=[y_min, y_max],
    title_text="Price"
)

fig.show()

##### Nobody knows where anything is going, worse, nobody knows what goes into edge and multiple things can be viable

- Lines on a chart aren't real structure, but they can be by accident
- Neither is a single tell in poker, but all that really matters is the outcome of that one event and the time average of your outcomes
- **we result aggressively** and have no idea what caused anything

###### ______________________________________________________________________________________________________________________________________

##### If anyone had a profitable strategy, they wouldn't share it online

One of the craziest things I've ever hear propogated by "professionals" in the space

Look at this bar, a one hour bar on AAPL, how many entry or exit combinations roughly are there?

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- Parameters ---
np.random.seed(np.random.randint(0, 10000))

symbol = "AAPL"
S0 = 190.00

n_seconds = 3600
right_candle_seconds = 30
animation_ms = 1000
hourly_vol = 0.006

# --- Generate 1-second ticks ---
dt = 1 / n_seconds

rets = np.random.normal(
    loc=0,
    scale=hourly_vol * np.sqrt(dt),
    size=n_seconds
)

prices = S0 * np.exp(np.cumsum(rets))
prices = np.insert(prices, 0, S0)

times = pd.date_range(
    start="2025-01-02 10:00:00",
    periods=n_seconds + 1,
    freq="s"
)

tick_df = pd.DataFrame({"price": prices}, index=times)

# --- Right-side micro candles ---
micro = tick_df["price"].resample(f"{right_candle_seconds}s").ohlc().dropna()

# --- Left-side 1H candle ---
open_price = prices[0]
high_price = prices.max()
low_price = prices.min()
close_price = prices[-1]

# --- Trade combination count ---
n_trade_paths = n_seconds * (n_seconds - 1) // 2

# --- Colors ---
off_white = "#e0e0e0"

# --- Figure ---
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        f"{symbol} One 1H Candle",
        f"{symbol} {right_candle_seconds}-Second Candles Inside That Bar"
    ),
    horizontal_spacing=0.12
)

# Trace 0: Left fixed 1H candle
fig.add_trace(
    go.Candlestick(
        x=[times[0]],
        open=[open_price],
        high=[high_price],
        low=[low_price],
        close=[close_price],
        increasing_line_color="#00ff88",
        decreasing_line_color="#ff4b4b",
        increasing_fillcolor="#00ff88",
        decreasing_fillcolor="#ff4b4b",
        name="1H Candle"
    ),
    row=1,
    col=1
)

# Trace 1: Right animated micro candles
fig.add_trace(
    go.Candlestick(
        x=[micro.index[0]],
        open=[micro["open"].iloc[0]],
        high=[micro["high"].iloc[0]],
        low=[micro["low"].iloc[0]],
        close=[micro["close"].iloc[0]],
        increasing_line_color="#00ff88",
        decreasing_line_color="#ff4b4b",
        increasing_fillcolor="#00ff88",
        decreasing_fillcolor="#ff4b4b",
        name="Micro Candles"
    ),
    row=1,
    col=2
)

# --- Frames ---
frames = []

for i in range(1, len(micro) + 1):
    frames.append(
        go.Frame(
            data=[
                go.Candlestick(
                    x=micro.index[:i],
                    open=micro["open"].iloc[:i],
                    high=micro["high"].iloc[:i],
                    low=micro["low"].iloc[:i],
                    close=micro["close"].iloc[:i],
                    increasing_line_color="#00ff88",
                    decreasing_line_color="#ff4b4b",
                    increasing_fillcolor="#00ff88",
                    decreasing_fillcolor="#ff4b4b"
                )
            ],
            traces=[1],
            name=f"f{i}"
        )
    )

fig.frames = frames

# --- Layout ---
min_y = low_price * 0.997
max_y = high_price * 1.003
frame_duration = animation_ms / len(frames)

fig.update_layout(
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=650,
    width=1200,
    margin=dict(t=100, b=80),
    showlegend=False,
    title=dict(
        text=(
            f"{symbol} 1H Candle • "
            f"{n_trade_paths:,} Possible Entry / Exit Combinations"
        ),
        x=0.5,
        font=dict(size=24)
    ),
    updatemenus=[
        {
            "type": "buttons",
            "buttons": [
                {
                    "label": "▶ Play",
                    "method": "animate",
                    "args": [
                        None,
                        {
                            "frame": {
                                "duration": frame_duration,
                                "redraw": True
                            },
                            "transition": {"duration": 0},
                            "fromcurrent": True,
                            "mode": "immediate"
                        }
                    ]
                },
                {
                    "label": "⏸ Pause",
                    "method": "animate",
                    "args": [
                        [None],
                        {
                            "frame": {
                                "duration": 0,
                                "redraw": True
                            },
                            "mode": "immediate",
                            "fromcurrent": True
                        }
                    ]
                }
            ],
            "direction": "left",
            "showactive": False,
            "x": 0.5,
            "xanchor": "center",
            "y": -0.08,
            "yanchor": "top"
        }
    ]
)

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.1)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white)
)

fig.update_xaxes(
    axis_style,
    title_text="One 1H Candle",
    rangeslider_visible=False,
    row=1,
    col=1
)

fig.update_yaxes(
    axis_style,
    range=[min_y, max_y],
    title_text="Price",
    row=1,
    col=1
)

fig.update_xaxes(
    axis_style,
    range=[micro.index[0], micro.index[-1]],
    title_text=f"{right_candle_seconds}-Second Candles",
    rangeslider_visible=False,
    row=1,
    col=2
)

fig.update_yaxes(
    axis_style,
    range=[min_y, max_y],
    title_text="Price",
    row=1,
    col=2
)

fig.show()

If I share knowledge about different strategy classes, or even active risk I am taking and managing you have literally no idea when and at what price I got in

Even worse think of a backtest, you use a one hour bar for your data but look at where the price moved **WHEN** should you get in?

Really you should use different bars TWAP/VWAP etc... but if this already doesn't make sense you are cooked from a retail perspective

###### ______________________________________________________________________________________________________________________________________

##### The Problem with Trading Coaches and the Notion of Alpha

Trading coaches *can be correct* but that doesn't mean you will make money

Think of it like hiring a poker coach, he isn't going to play hands for you...

The entire industry feels like a scam because people **DON'T UNDERSTAND MATH**

##### What if he is correct and on average there is some edge, you can't prove that and nobody can

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- Parameters ---
np.random.seed(np.random.randint(0, 10000))

symbol = "AAPL"
S0 = 190.00

n_seconds = 3600
right_candle_seconds = 30
animation_ms = 1000
hourly_vol = 0.006

# --- Generate 1-second ticks ---
dt = 1 / n_seconds

rets = np.random.normal(
    loc=0,
    scale=hourly_vol * np.sqrt(dt),
    size=n_seconds
)

prices = S0 * np.exp(np.cumsum(rets))
prices = np.insert(prices, 0, S0)

times = pd.date_range(
    start="2025-01-02 10:00:00",
    periods=n_seconds + 1,
    freq="s"
)

tick_df = pd.DataFrame({"price": prices}, index=times)

# --- Right-side micro candles ---
micro = tick_df["price"].resample(f"{right_candle_seconds}s").ohlc().dropna()

# --- Left-side 1H candle ---
open_price = prices[0]
high_price = prices.max()
low_price = prices.min()
close_price = prices[-1]

# --- Trade combination count ---
n_trade_paths = n_seconds * (n_seconds - 1) // 2

# --- Colors ---
off_white = "#e0e0e0"

# --- Figure ---
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        f"{symbol} One 1H Candle",
        f"{symbol} {right_candle_seconds}-Second Candles Inside That Bar"
    ),
    horizontal_spacing=0.12
)

# Trace 0: Left fixed 1H candle
fig.add_trace(
    go.Candlestick(
        x=[times[0]],
        open=[open_price],
        high=[high_price],
        low=[low_price],
        close=[close_price],
        increasing_line_color="#00ff88",
        decreasing_line_color="#ff4b4b",
        increasing_fillcolor="#00ff88",
        decreasing_fillcolor="#ff4b4b",
        name="1H Candle"
    ),
    row=1,
    col=1
)

# Trace 1: Right animated micro candles
fig.add_trace(
    go.Candlestick(
        x=[micro.index[0]],
        open=[micro["open"].iloc[0]],
        high=[micro["high"].iloc[0]],
        low=[micro["low"].iloc[0]],
        close=[micro["close"].iloc[0]],
        increasing_line_color="#00ff88",
        decreasing_line_color="#ff4b4b",
        increasing_fillcolor="#00ff88",
        decreasing_fillcolor="#ff4b4b",
        name="Micro Candles"
    ),
    row=1,
    col=2
)

# --- Frames ---
frames = []

for i in range(1, len(micro) + 1):
    frames.append(
        go.Frame(
            data=[
                go.Candlestick(
                    x=micro.index[:i],
                    open=micro["open"].iloc[:i],
                    high=micro["high"].iloc[:i],
                    low=micro["low"].iloc[:i],
                    close=micro["close"].iloc[:i],
                    increasing_line_color="#00ff88",
                    decreasing_line_color="#ff4b4b",
                    increasing_fillcolor="#00ff88",
                    decreasing_fillcolor="#ff4b4b"
                )
            ],
            traces=[1],
            name=f"f{i}"
        )
    )

fig.frames = frames

# --- Layout ---
min_y = low_price * 0.997
max_y = high_price * 1.003
frame_duration = animation_ms / len(frames)

fig.update_layout(
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=650,
    width=1200,
    margin=dict(t=100, b=80),
    showlegend=False,
    title=dict(
        text=(
            f"{symbol} 1H Candle • "
            f"{n_trade_paths:,} Possible Entry / Exit Combinations"
        ),
        x=0.5,
        font=dict(size=24)
    ),
    updatemenus=[
        {
            "type": "buttons",
            "buttons": [
                {
                    "label": "▶ Play",
                    "method": "animate",
                    "args": [
                        None,
                        {
                            "frame": {
                                "duration": frame_duration,
                                "redraw": True
                            },
                            "transition": {"duration": 0},
                            "fromcurrent": True,
                            "mode": "immediate"
                        }
                    ]
                },
                {
                    "label": "⏸ Pause",
                    "method": "animate",
                    "args": [
                        [None],
                        {
                            "frame": {
                                "duration": 0,
                                "redraw": True
                            },
                            "mode": "immediate",
                            "fromcurrent": True
                        }
                    ]
                }
            ],
            "direction": "left",
            "showactive": False,
            "x": 0.5,
            "xanchor": "center",
            "y": -0.08,
            "yanchor": "top"
        }
    ]
)

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.1)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white)
)

fig.update_xaxes(
    axis_style,
    title_text="One 1H Candle",
    rangeslider_visible=False,
    row=1,
    col=1
)

fig.update_yaxes(
    axis_style,
    range=[min_y, max_y],
    title_text="Price",
    row=1,
    col=1
)

fig.update_xaxes(
    axis_style,
    range=[micro.index[0], micro.index[-1]],
    title_text=f"{right_candle_seconds}-Second Candles",
    rangeslider_visible=False,
    row=1,
    col=2
)

fig.update_yaxes(
    axis_style,
    range=[min_y, max_y],
    title_text="Price",
    row=1,
    col=2
)

fig.show()

###### ______________________________________________________________________________________________________________________________________

#### The real problem: implied probabilities collapse one time

Probabilities collapse 1 time, you don't understand statistics enough to see how important this is, we don't get asymptotics in real life


**Positive EV over TIME may not be positive EV for a single event and we will never know**

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# --- Parameters ---
np.random.seed(np.random.randint(0, 10000))

symbol = "AAPL"
S0 = 190.00

n_paths = 10
n_seconds = 3600
animation_ms = 1000
hourly_vol = 0.006

earnings_second = n_seconds // 2
gap_up_path = np.random.randint(0, n_paths)

earnings_jumps = np.full(n_paths, -0.01)
earnings_jumps[gap_up_path] = 0.01

times = pd.date_range(
    start="2025-01-02 10:00:00",
    periods=n_seconds + 1,
    freq="s"
)

earnings_time = times[earnings_second]

# --- Simulate generic paths ---
all_paths = []
dt = 1 / n_seconds

for path_id in range(n_paths):
    rets = np.random.normal(
        loc=0,
        scale=hourly_vol * np.sqrt(dt),
        size=n_seconds
    )

    prices = S0 * np.exp(np.cumsum(rets))
    prices = np.insert(prices, 0, S0)

    prices[earnings_second:] *= (1 + earnings_jumps[path_id])

    all_paths.append(prices)

all_paths = np.array(all_paths)

# --- Animation frames ---
frame_skip = 20
frame_indices = list(range(1, n_seconds + 1, frame_skip))

if frame_indices[-1] != n_seconds:
    frame_indices.append(n_seconds)

# --- Figure ---
fig = go.Figure()

for path_id in range(n_paths):
    is_gap_up = path_id == gap_up_path

    fig.add_trace(
        go.Scatter(
            x=[times[0]],
            y=[all_paths[path_id, 0]],
            mode="lines",
            line=dict(
                color="#00ff88" if is_gap_up else "#ff4b4b",
                width=5 if is_gap_up else 2
            ),
            opacity=1.0 if is_gap_up else 0.25,
            showlegend=False
        )
    )

# --- Frames and slider ---
frames = []
slider_steps = []

for frame_num, i in enumerate(frame_indices):
    frame_name = f"frame_{frame_num}"

    frames.append(
        go.Frame(
            data=[
                go.Scatter(
                    x=times[:i + 1],
                    y=all_paths[path_id, :i + 1],
                    mode="lines",
                    line=dict(
                        color="#00ff88" if path_id == gap_up_path else "#ff4b4b",
                        width=5 if path_id == gap_up_path else 2
                    ),
                    opacity=1.0 if path_id == gap_up_path else 0.25
                )
                for path_id in range(n_paths)
            ],
            traces=list(range(n_paths)),
            name=frame_name
        )
    )

    slider_steps.append(
        {
            "args": [
                [frame_name],
                {
                    "frame": {"duration": 0, "redraw": True},
                    "mode": "immediate",
                    "fromcurrent": True
                }
            ],
            "label": str(i),
            "method": "animate"
        }
    )

fig.frames = frames

frame_duration = animation_ms / len(frames)

# --- Styling ---
off_white = "#e0e0e0"

y_min = all_paths.min() * 0.995
y_max = all_paths.max() * 1.005

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.1)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white)
)

fig.update_layout(
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=720,
    width=1100,
    margin=dict(t=100, b=170),
    showlegend=False,
    title=dict(
        text=f"{symbol} Simulated Earnings Paths • 1 Gap Up +1% / {n_paths - 1} Gap Down -1%",
        x=0.5,
        font=dict(size=23)
    ),
    shapes=[
        dict(
            type="line",
            x0=earnings_time,
            x1=earnings_time,
            y0=y_min,
            y1=y_max,
            xref="x",
            yref="y",
            line=dict(color="#ffffff", width=2, dash="dash")
        )
    ],
    annotations=[
        dict(
            x=earnings_time,
            y=y_max,
            xref="x",
            yref="y",
            text="Earnings",
            showarrow=False,
            yanchor="bottom",
            font=dict(color="#ffffff", size=14)
        )
    ],
    updatemenus=[
        {
            "type": "buttons",
            "buttons": [
                {
                    "label": "▶ Play",
                    "method": "animate",
                    "args": [
                        None,
                        {
                            "frame": {"duration": frame_duration, "redraw": True},
                            "transition": {"duration": 0},
                            "fromcurrent": True,
                            "mode": "immediate"
                        }
                    ]
                },
                {
                    "label": "⏸ Pause",
                    "method": "animate",
                    "args": [
                        [None],
                        {
                            "frame": {"duration": 0, "redraw": True},
                            "mode": "immediate",
                            "fromcurrent": True
                        }
                    ]
                }
            ],
            "direction": "left",
            "showactive": False,
            "x": 0.12,
            "xanchor": "left",
            "y": -0.18,
            "yanchor": "top"
        }
    ],
    sliders=[
        {
            "active": 0,
            "yanchor": "top",
            "xanchor": "left",
            "currentvalue": {
                "font": {"size": 13, "color": off_white},
                "prefix": "Second: ",
                "visible": True,
                "xanchor": "right"
            },
            "transition": {"duration": 0},
            "pad": {"b": 10, "t": 45},
            "len": 0.78,
            "x": 0.18,
            "y": -0.18,
            "steps": slider_steps
        }
    ]
)

fig.update_xaxes(
    axis_style,
    range=[times[0], times[-1]],
    title_text="One-Hour Earnings Window"
)

fig.update_yaxes(
    axis_style,
    range=[y_min, y_max],
    title_text="Price"
)

fig.show()

This is why it is dangerous, you can justify wins and losses by anything and you will never know if you were right

###### ______________________________________________________________________________________________________________________________________

##### Arbitrary Strategy Criticisms

Seriously guys, not everything is a cross-sectional average

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# --- Parameters ---

symbol = "AAPL"
S0 = 190.00
strike_price = 191.00

n_paths = 10
n_seconds = 3600
animation_ms = 1000
hourly_vol = 0.006

earnings_second = n_seconds // 2
gap_up_path = np.random.randint(0, n_paths)

earnings_jumps = np.full(n_paths, -0.01)
earnings_jumps[gap_up_path] = 0.01

times = pd.date_range(
    start="2025-01-02 10:00:00",
    periods=n_seconds + 1,
    freq="s"
)

earnings_time = times[earnings_second]

# --- Simulate paths ---
all_paths = []
dt = 1 / n_seconds

for path_id in range(n_paths):
    rets = np.random.normal(
        loc=0,
        scale=hourly_vol * np.sqrt(dt),
        size=n_seconds
    )

    prices = S0 * np.exp(np.cumsum(rets))
    prices = np.insert(prices, 0, S0)

    prices[earnings_second:] *= (1 + earnings_jumps[path_id])

    all_paths.append(prices)

all_paths = np.array(all_paths)

# --- Expiry outcomes ---
final_prices = all_paths[:, -1]
is_itm = final_prices > strike_price
intrinsic_values = np.maximum(final_prices - strike_price, 0)

# --- Animation setup ---
frame_skip = 20
frame_indices = list(range(1, n_seconds + 1, frame_skip))

if frame_indices[-1] != n_seconds:
    frame_indices.append(n_seconds)

# --- Figure ---
fig = go.Figure()

for path_id in range(n_paths):
    path_itm = is_itm[path_id]

    fig.add_trace(
        go.Scatter(
            x=[times[0]],
            y=[all_paths[path_id, 0]],
            mode="lines",
            line=dict(
                color="#00ff88" if path_itm else "#ff4b4b",
                width=5 if path_itm else 2
            ),
            opacity=1.0 if path_itm else 0.25,
            showlegend=False
        )
    )

# --- Styling ---
off_white = "#e0e0e0"

y_min = min(all_paths.min(), strike_price) * 0.995
y_max = max(all_paths.max(), strike_price) * 1.005

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.1)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white)
)

base_annotations = [
    dict(
        x=earnings_time,
        y=y_max,
        xref="x",
        yref="y",
        text="Earnings",
        showarrow=False,
        yanchor="bottom",
        font=dict(color="#ffffff", size=14)
    ),
    dict(
        x=times[20],
        y=strike_price,
        xref="x",
        yref="y",
        text=f"Strike ${strike_price:.2f}",
        showarrow=False,
        font=dict(color="#ffd166", size=13),
        bgcolor="rgba(0,0,0,0.45)"
    )
]

# --- Frames and slider ---
frames = []
slider_steps = []

for frame_num, i in enumerate(frame_indices):
    frame_name = f"frame_{frame_num}"

    frame_annotations = list(base_annotations)

    if i == n_seconds:
        for path_id in range(n_paths):
            final_price = final_prices[path_id]
            path_itm = is_itm[path_id]
            intrinsic = intrinsic_values[path_id]

            frame_annotations.append(
                dict(
                    x=times[-1],
                    y=final_price,
                    xref="x",
                    yref="y",
                    text=(
                        f"ITM +${intrinsic:.2f}"
                        if path_itm
                        else "Expired Worthless"
                    ),
                    showarrow=False,
                    xanchor="left",
                    font=dict(
                        color="#00ff88" if path_itm else "#ff4b4b",
                        size=12
                    ),
                    bgcolor="rgba(0,0,0,0.65)"
                )
            )

    frames.append(
        go.Frame(
            data=[
                go.Scatter(
                    x=times[:i + 1],
                    y=all_paths[path_id, :i + 1],
                    mode="lines",
                    line=dict(
                        color="#00ff88" if is_itm[path_id] else "#ff4b4b",
                        width=5 if is_itm[path_id] else 2
                    ),
                    opacity=1.0 if is_itm[path_id] else 0.25
                )
                for path_id in range(n_paths)
            ],
            traces=list(range(n_paths)),
            layout=go.Layout(annotations=frame_annotations),
            name=frame_name
        )
    )

    slider_steps.append(
        {
            "args": [
                [frame_name],
                {
                    "frame": {"duration": 0, "redraw": True},
                    "mode": "immediate",
                    "fromcurrent": True
                }
            ],
            "label": str(i),
            "method": "animate"
        }
    )

fig.frames = frames

frame_duration = animation_ms / len(frames)

# --- Layout ---
fig.update_layout(
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=720,
    width=1100,
    margin=dict(t=100, b=170, r=150),
    showlegend=False,
    title=dict(
        text=(
            f"{symbol} Simulated Earnings Paths • "
            f"${strike_price:.2f} Call Strike • "
            f"{is_itm.sum()} ITM / {n_paths - is_itm.sum()} Worthless"
        ),
        x=0.5,
        font=dict(size=23)
    ),
    shapes=[
        dict(
            type="line",
            x0=earnings_time,
            x1=earnings_time,
            y0=y_min,
            y1=y_max,
            xref="x",
            yref="y",
            line=dict(color="#ffffff", width=2, dash="dash")
        ),
        dict(
            type="line",
            x0=times[0],
            x1=times[-1],
            y0=strike_price,
            y1=strike_price,
            xref="x",
            yref="y",
            line=dict(color="#ffd166", width=2, dash="dot")
        )
    ],
    annotations=base_annotations,
    updatemenus=[
        {
            "type": "buttons",
            "buttons": [
                {
                    "label": "▶ Play",
                    "method": "animate",
                    "args": [
                        None,
                        {
                            "frame": {"duration": frame_duration, "redraw": True},
                            "transition": {"duration": 0},
                            "fromcurrent": True,
                            "mode": "immediate"
                        }
                    ]
                },
                {
                    "label": "⏸ Pause",
                    "method": "animate",
                    "args": [
                        [None],
                        {
                            "frame": {"duration": 0, "redraw": True},
                            "mode": "immediate",
                            "fromcurrent": True
                        }
                    ]
                }
            ],
            "direction": "left",
            "showactive": False,
            "x": 0.12,
            "xanchor": "left",
            "y": -0.18,
            "yanchor": "top"
        }
    ],
    sliders=[
        {
            "active": 0,
            "yanchor": "top",
            "xanchor": "left",
            "currentvalue": {
                "font": {"size": 13, "color": off_white},
                "prefix": "Second: ",
                "visible": True,
                "xanchor": "right"
            },
            "transition": {"duration": 0},
            "pad": {"b": 10, "t": 45},
            "len": 0.78,
            "x": 0.18,
            "y": -0.18,
            "steps": slider_steps
        }
    ]
)

fig.update_xaxes(
    axis_style,
    range=[times[0], times[-1]],
    title_text="One-Hour Earnings Window"
)

fig.update_yaxes(
    axis_style,
    range=[y_min, y_max],
    title_text="Price"
)

fig.show()

---

#### 💭 Closing Thoughts and Future Topics
 
**Future Topics**

Technical Videos and Other Discussions

 - Fama-French / Carhart and Factor Modeling in General
 - Hawkes Processes
 - Merton Jump Diffusion Model (and Characteristic Function Pricing, Carr-Madan 1999)
 - Market-Making Models and Simulation (Stoikov-Avellaneda)
 - My First Year as a Quant
 - Why Hedge Funds are Actually Secretive
 - Non-Markovian Models (fractional Brownian motion, Volterra Process)
 - Top 3 Uses of Linear Algebra for Quant Finance
 - Girsanov's Change of Measure
 - Rough Path Theory, Applications of Path Signatures
 - Sig-Vol Model, Calibration, and Pricing
 - Trading with Alternative Data Sources
 - Pairs Trading and Statistical Arbitrage
 - Data Cleaning & Outlier Handling in Financial Time Series
 - Practical Issues in Multi-Asset Portfolio Backtesting
 - Risk Premia Harvesting: Equity, FX, Rates

[Ideas for Interactive Brokers Apps and Tutorials](https://www.interactivebrokers.com/mkt/?src=quantguildY&url=%2Fen%2Fwhyib%2Foverview.php)

- How Interactive Broker's API Works (EWrapper/EClient)
- How to Backtest a Trading Strategy with Interactive Brokers
- Algorithmic Volatility Trading System

---

####  $\text{Copyright © 2026 Quant Guild} \quad \quad \quad \quad \text{Author: Roman Paolucci}$